# `EXCEL_INGESTION_JOB` (엑셀 스프레드시트 구조화 및 pgvector 인덱싱)

- **Recipe File**: `jobs/excel_ingestion.py`
- **Job ID**: `excel_ingestion`
- **Target Queue**: `workflow-core`
- **Version**: `2`

## 1. 개요 및 파이프라인 흐름
비정형 재무제표 엑셀 파일(병합 셀, 다중 헤더, 주석 등)을 Luna VLM 모델로 시각 감지하고, 정형화된 셀 자연어 텍스트로 직렬화하여 pgvector HNSW 인덱스 및 시트/기업 메타데이터를 자동 적재하는 수집 DAG입니다.

```
  [source] (processed_file_selector)
     │
     ▼
  [structure] (luna_vlm_structure_detector) ───────┐
     │                                             │
     ▼                                             ▼
  [serialize] (cell_text_serializer)      [persist-sheets] (sheet_metadata_persistence)
     │                                             ▲
     ▼                                             │
  [embed] (cell_text_embedder)                     │
     │                                             │
     ▼                                             │
  [write-index] (pgvector_index_writer) ───────────┴────► [persist-company] (company_entity_extractor)
```

In [ ]:
import sys
from pathlib import Path
import json

# Find repository root by walking up from cwd to find jobs/__init__.py
current = Path.cwd()
PROJECT_ROOT = None
for parent in [current] + list(current.parents):
    if (parent / "jobs" / "__init__.py").exists():
        PROJECT_ROOT = parent
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find repository root containing jobs/__init__.py")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jobs.excel_ingestion import EXCEL_INGESTION_JOB

print(f"📌 Loaded Job: {EXCEL_INGESTION_JOB.name} ({EXCEL_INGESTION_JOB.job_id})")
print(f"Description: {EXCEL_INGESTION_JOB.description}")
print(f"\n--- Nodes ({len(EXCEL_INGESTION_JOB.nodes)}) ---")
for node in EXCEL_INGESTION_JOB.nodes:
    print(f"  - Node: {node.node_id:<16} | Module: {node.module_type}")

print(f"\n--- Edges ({len(EXCEL_INGESTION_JOB.edges)}) ---")
for edge in EXCEL_INGESTION_JOB.edges:
    print(f"  - [{edge.source}.{edge.source_output}] -> [{edge.target}.{edge.target_input}]")